2026/09/27 01:02:09 [642163] Number of files: 226,076 (reg: 162,412, dir: 63,543, link: 121)

2026/09/27 01:02:09 [642163] Number of created files: 7,246 (reg: 6,815, dir: 393, link: 38)

2026/09/27 01:02:09 [642163] Number of deleted files: 0

2026/09/27 01:02:09 [642163] Number of regular files transferred: 29,741

2026/09/27 01:02:09 [642163] Total file size: 204,937,456,429 bytes

2026/09/27 01:02:09 [642163] Total transferred file size: 1,541,261,548 bytes

2026/09/27 01:02:09 [642163] Literal data: 644,656,632 bytes

2026/09/27 01:02:09 [642163] Matched data: 896,604,916 bytes

2026/09/27 01:02:09 [642163] File list size: 655,275

2026/09/27 01:02:09 [642163] File list generation time: 0.001 seconds

2026/09/27 01:02:09 [642163] File list transfer time: 0.000 seconds

2026/09/27 01:02:09 [642163] Total bytes sent: 208,961,618

2026/09/27 01:02:09 [642163] Total bytes received: 4,317,865

2026/09/27 01:02:09 [642163] sent 208,961,618 bytes  received 4,317,865 bytes  1,659,762.51 bytes/sec

2026/09/27 01:02:09 [642163] total size is 204,937,456,429  speedup is 960.89

Backup completed successfully at Sun 27 Sep 01:02:09 CEST 2026

=== Backup finished at Sun 27 Sep 01:02:09 CEST 2026 ===


In [11]:
import java.time.LocalDateTime
import java.time.ZonedDateTime
import java.time.format.DateTimeFormatter
import java.time.Duration
import java.util.Locale

data class FileDistribution(
    val regularFiles: Int,
    val directories: Int,
    val fileLinks: Int,
)

data class Backup(
    val uuid: String = "", //UUID.randomUUID().toString(),
    val datetime: LocalDateTime = LocalDateTime.now(),
    val fileName: String = "",
    val success: Boolean? = null,
    val duration: Int? = null,
    val totalFiles: FileDistribution? = null,
    val createdFiles: FileDistribution? = null,
    val deletedFiles: FileDistribution? = null,
    val transferredRegularFiles: Int? = null,
    val totalFileSize: Long? = null,
    val transferredFileSize: Long? = null,
    val totalBytesSent: Long? = null,
    val totalBytesReceived: Long? = null,
    val bytesPerSecond: Int? = null,
    val speedup: Int? = null,
)

/**
 * @param logText consists of the first line with the timestamp of the start and then the last 17 lines which contain the statistics after a successful backup
 */
fun parseRsyncLog(logText: String): Backup {
    val isFailed = logText.contains("Backup failed")
    val lines = logText.lines().filter { it.isNotBlank() }

    // Helper to extract dates from either "2026/09/27 01:02:09" OR "Sun 27 Sep 01:02:09 CEST 2026"
    fun extractDate(line: String?): LocalDateTime? {
        if (line == null) return null
        val match = """([A-Za-z]{3}\s+\d{1,2}\s+[A-Za-z]{3}\s+\d{2}:\d{2}:\d{2}\s+[A-Za-z]{3,4}\s+\d{4})""".toRegex().find(line)
        if (match != null) {
            val fmt = DateTimeFormatter.ofPattern("EEE d MMM HH:mm:ss z yyyy", Locale.ENGLISH)
            try { return ZonedDateTime.parse(match.value, fmt).toLocalDateTime() } catch (e: Exception) {}
        }
        return null
    }

    // 1. Parse Datetime (Strictly from the first line)
    val parsedDatetime = extractDate(lines.firstOrNull())

    // 2. Parse Duration (Difference between first and last line)
    val endDatetime = extractDate(lines.last())
    val computedDuration = if (parsedDatetime != null && endDatetime != null) {
        Duration.between(parsedDatetime, endDatetime).seconds.toInt()
    } else null

    val fileFormatter = DateTimeFormatter.ofPattern("yyyy-MM-dd_HH-mm")
    val fileName = "${endDatetime!!.format(fileFormatter)}_log-backup.txt"
    // 3. Early return if backup failed
    if (isFailed) {
        return Backup(
            fileName = fileName,
            datetime = parsedDatetime ?: LocalDateTime.now(),
            success = false,
            duration = computedDuration
        )
    }

    // 4. Full parse if backup succeeded
    fun String.toCleanLong(): Long = this.replace(",", "").toLongOrNull() ?: 0L
    fun String.toCleanInt(): Int = this.replace(",", "").substringBefore('.').toIntOrNull() ?: 0

    var totalFiles: FileDistribution? = null
    var createdFiles: FileDistribution? = null
    var deletedFiles: FileDistribution? = null
    var transRegFiles: Int? = null
    var totalSize: Long? = null
    var transSize: Long? = null
    var bytesSent: Long? = null
    var bytesReceived: Long? = null
    var bytesPerSec: Int? = null
    var parsedSpeedup: Int? = null

    val distributionRegex = """\(reg:\s*([\d,]+),\s*dir:\s*([\d,]+),\s*link:\s*([\d,]+)\)""".toRegex()

    fun parseFileDistribution(line: String): FileDistribution? {
        val match = distributionRegex.find(line)
        if (match != null) {
            return FileDistribution(match.groupValues[1].toCleanInt(), match.groupValues[2].toCleanInt(), match.groupValues[3].toCleanInt())
        }
        val singleNumberMatch = """:\s*([\d,]+)$""".toRegex().find(line)
        val count = singleNumberMatch?.groupValues?.get(1)?.toCleanInt() ?: 0
        return if (count == 0) FileDistribution(0, 0, 0) else null
    }

    lines.forEach { line ->
        when {
            line.contains("Number of files:") -> totalFiles = parseFileDistribution(line)
            line.contains("Number of created files:") -> createdFiles = parseFileDistribution(line)
            line.contains("Number of deleted files:") -> deletedFiles = parseFileDistribution(line)
            line.contains("Number of regular files transferred:") ->
                transRegFiles = """transferred:\s*([\d,]+)""".toRegex().find(line)?.groupValues?.get(1)?.toCleanInt()
            line.contains("Total file size:") ->
                totalSize = """size:\s*([\d,]+)""".toRegex().find(line)?.groupValues?.get(1)?.toCleanLong()
            line.contains("Total transferred file size:") ->
                transSize = """size:\s*([\d,]+)""".toRegex().find(line)?.groupValues?.get(1)?.toCleanLong()
            line.contains("bytes/sec") -> {
                val match = """sent\s+([\d,]+)\s+bytes\s+received\s+([\d,]+)\s+bytes\s+([\d.,]+)\s+bytes/sec""".toRegex().find(line)
                match?.let {
                    bytesSent = it.groupValues[1].toCleanLong()
                    bytesReceived = it.groupValues[2].toCleanLong()
                    bytesPerSec = it.groupValues[3].toCleanInt()
                }
            }
            line.contains("speedup is") ->
                parsedSpeedup = """speedup is\s+([\d.,]+)""".toRegex().find(line)?.groupValues?.get(1)?.toCleanInt()
        }
    }

    return Backup(
        fileName = fileName,
        datetime = parsedDatetime ?: LocalDateTime.now(),
        success = true,
        duration = computedDuration,
        totalFiles = totalFiles,
        createdFiles = createdFiles,
        deletedFiles = deletedFiles,
        transferredRegularFiles = transRegFiles,
        totalFileSize = totalSize,
        transferredFileSize = transSize,
        totalBytesSent = bytesSent,
        totalBytesReceived = bytesReceived,
        bytesPerSecond = bytesPerSec,
        speedup = parsedSpeedup
    )
}

val rawLog = """
    === Starting backup at Sun 27 Sep 01:02:00 CEST 2026 ===
    2026/09/27 01:02:09 [642163] Number of files: 226,076 (reg: 162,412, dir: 63,543, link: 121)
    2026/09/27 01:02:09 [642163] Number of deleted files: 0
    Backup failed at Sun 27 Sep 01:02:15 CEST 2026
""".trimIndent()

val rawLog2 = """
    === Backup finished at Sun 27 Sep 01:00:00 CEST 2026 ===
    2026/09/27 01:02:09 [642163] Number of files: 226,076 (reg: 162,412, dir: 63,543, link: 121)
    2026/09/27 01:02:09 [642163] Number of created files: 7,246 (reg: 6,815, dir: 393, link: 38)
    2026/09/27 01:02:09 [642163] Number of deleted files: 0
    2026/09/27 01:02:09 [642163] Number of regular files transferred: 29,741
    2026/09/27 01:02:09 [642163] Total file size: 204,937,456,429 bytes
    2026/09/27 01:02:09 [642163] Total transferred file size: 1,541,261,548 bytes
    2026/09/27 01:02:09 [642163] Literal data: 644,656,632 bytes
    2026/09/27 01:02:09 [642163] Matched data: 896,604,916 bytes
    2026/09/27 01:02:09 [642163] File list size: 655,275
    2026/09/27 01:02:09 [642163] File list generation time: 0.001 seconds
    2026/09/27 01:02:09 [642163] File list transfer time: 0.000 seconds
    2026/09/27 01:02:09 [642163] Total bytes sent: 208,961,618
    2026/09/27 01:02:09 [642163] Total bytes received: 4,317,865
    2026/09/27 01:02:09 [642163] sent 208,961,618 bytes received 4,317,865 bytes 1,659,762.51 bytes/sec
    2026/09/27 01:02:09 [642163] total size is 204,937,456,429 speedup is 960.89
    Backup completed successfully at Sun 27 Sep 01:02:09 CEST 2026
    === Backup finished at Sun 27 Sep 01:02:09 CEST 2026 ===
""".trimIndent()

val backup = parseRsyncLog(rawLog)
val backup2 = parseRsyncLog(rawLog2)
backup2

Backup(uuid=, datetime=2026-09-27T01:00, fileName=2026-09-27_01-02_log-backup.txt, success=true, duration=129, totalFiles=FileDistribution(regularFiles=162412, directories=63543, fileLinks=121), createdFiles=FileDistribution(regularFiles=6815, directories=393, fileLinks=38), deletedFiles=FileDistribution(regularFiles=0, directories=0, fileLinks=0), transferredRegularFiles=29741, totalFileSize=204937456429, transferredFileSize=1541261548, totalBytesSent=208961618, totalBytesReceived=4317865, bytesPerSecond=1659762, speedup=960)